# HDB resale — RandomForest v1

**Model:** `RandomForestRegressor` on **`log1p(resale_price)`**; metrics and submissions use **`expm1`** (dollar scale).

**Validation (two runs):**
1. **Time-based:** hold out the **last 12 calendar months** (`Tranc_YearMonth`).
2. **Random:** **80% / 20%** `train_test_split` (`shuffle=True`, `random_state=RNG`).

**Tuning:** separate `RandomizedSearchCV` runs for time and random splits using `neg_root_mean_squared_error`.

**Submissions:**
- `ROOT / submission / sub_rf_v1_t.csv` (time split)
- `ROOT / submission / sub_rf_v1_r.csv` (random split)



In [1]:
# Paths
from pathlib import Path

import numpy as np
import pandas as pd

_cwd = Path.cwd()
ROOT = _cwd.parent if _cwd.name == "notebook" else _cwd
TRAIN_PATH = ROOT / "data" / "train.csv"
TEST_PATH = ROOT / "data" / "test.csv"
SAMPLE_SUB_PATH = ROOT / "data" / "sample_sub_reg.csv"
SUBMISSION_PATH_T = ROOT / "submission" / "sub_rf_v1_t.csv"
SUBMISSION_PATH_R = ROOT / "submission" / "sub_rf_v1_r.csv"

RNG = 42

print(f"ROOT: {ROOT.resolve()}")
train = pd.read_csv(TRAIN_PATH, low_memory=False)
test = pd.read_csv(TEST_PATH, low_memory=False)
print(train.shape, test.shape)



ROOT: /Users/ian/Documents/NTU/DSAI/Module 3/HDB Kaggle
(150634, 77) (16735, 76)


In [2]:
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.ensemble import RandomForestRegressor

try:
    from sklearn.metrics import root_mean_squared_error
except ImportError:
    def root_mean_squared_error(y_true, y_pred):
        return mean_squared_error(y_true, y_pred, squared=False)

TARGET = "resale_price"



In [3]:
ROOMS_FROM_FLAT = {
    "1 ROOM": 1,
    "2 ROOM": 2,
    "3 ROOM": 3,
    "4 ROOM": 4,
    "5 ROOM": 5,
    "EXECUTIVE": 6,
    "MULTI-GENERATION": 7,
}


def add_engineered_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    pc = out["postal"].astype(str).str.replace(r"\.0$", "", regex=True)
    out["postal_sector"] = pd.to_numeric(pc.str.slice(0, 2), errors="coerce")

    ms = pd.to_numeric(out["mid_storey"], errors="coerce")
    mx = pd.to_numeric(out["max_floor_lvl"], errors="coerce")
    out["storey_ratio"] = np.where(mx > 0, ms / mx, np.nan)

    rcols = ["1room_rental", "2room_rental", "3room_rental", "other_room_rental"]
    total_rent = np.zeros(len(out))
    for c in rcols:
        total_rent += pd.to_numeric(out[c], errors="coerce").fillna(0).to_numpy(dtype=float)
    td = pd.to_numeric(out["total_dwelling_units"], errors="coerce").to_numpy(dtype=float)
    out["rental_ratio"] = np.where(td > 0, total_rent / td, np.nan)

    out["rooms_num"] = out["flat_type"].map(ROOMS_FROM_FLAT).astype(float)
    ty = pd.to_numeric(out["Tranc_Year"], errors="coerce")
    tm = pd.to_numeric(out["Tranc_Month"], errors="coerce")
    out["month_index"] = (ty - 2000) * 12 + tm
    return out


train = add_engineered_features(train)
test = add_engineered_features(test)

DROP_FEATURES = [
    "id",
    "Tranc_YearMonth",
    "floor_area_sqft",
    "postal",
    "address",
    "block",
    "street_name",
    "flat_type",
    "flat_model",
    "1room_sold",
    "2room_sold",
    "3room_sold",
    "4room_sold",
    "5room_sold",
    "exec_sold",
    "multigen_sold",
    "studio_apartment_sold",
    "1room_rental",
    "2room_rental",
    "3room_rental",
    "other_room_rental",
    "bus_stop_name",
    "sec_sch_name",
]

feature_cols = [c for c in train.columns if c not in DROP_FEATURES and c != TARGET]
assert not set(feature_cols) - set(test.columns)

X_train = train[feature_cols].copy()
X_test = test[feature_cols].copy()
y = train[TARGET].astype(float)
period = pd.to_datetime(train["Tranc_YearMonth"], format="%Y-%m")

print(f"Features: {len(feature_cols)}")



Features: 58


In [4]:
def imputation_stats(X_ref: pd.DataFrame, cat_cols: list, num_cols: list):
    num_med = {
        c: pd.to_numeric(X_ref[c], errors="coerce").median()
        for c in num_cols
    }
    cat_fill = {}
    for c in cat_cols:
        s = X_ref[c].astype(str).replace("nan", np.nan)
        m = s.mode(dropna=True)
        cat_fill[c] = m.iloc[0] if len(m) else "_MISSING_"
    return num_med, cat_fill


def prepare_for_rf(
    X: pd.DataFrame,
    cat_cols: list,
    num_cols: list,
    num_med: dict,
    cat_fill: dict,
    cat_mapping: dict = None,
):
    # Impute with train stats and encode categoricals with train-fitted integer mapping.
    out = X.copy()

    for c in num_cols:
        v = pd.to_numeric(out[c], errors="coerce")
        out[c] = v.fillna(num_med[c]).astype(float)

    learned_mapping = {} if cat_mapping is None else cat_mapping

    for c in cat_cols:
        s = out[c].astype(str).replace("nan", np.nan).fillna(cat_fill[c])
        if cat_mapping is None:
            cats = pd.Index(pd.Series(s, dtype="string").dropna().unique()).sort_values()
            mapping = {v: i for i, v in enumerate(cats)}
            learned_mapping[c] = mapping
        mapping = learned_mapping[c]
        out[c] = s.map(mapping).fillna(-1).astype(float)

    if cat_mapping is None:
        return out, learned_mapping
    return out


num_cols = X_train.select_dtypes(include=["number", "bool"]).columns.tolist()
cat_cols = [c for c in feature_cols if c not in num_cols]
print(f"Numeric: {len(num_cols)}, categorical: {len(cat_cols)}")



Numeric: 47, categorical: 11


In [5]:
rf_param_dist = {
    "n_estimators": [400, 700, 1000],
    "max_depth": [None, 16, 24, 32],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", 0.5, 0.8],
    "bootstrap": [True],
}

rf_base = RandomForestRegressor(
    random_state=RNG,
    n_jobs=-1,
)



In [6]:
# --- A) Time-based validation + tuning (last 12 months) ---
cutoff = period.max() - pd.DateOffset(months=12)
tr_time = period < cutoff
va_time = ~tr_time
X_tr_raw_t = X_train.loc[tr_time]
X_va_raw_t = X_train.loc[va_time]
y_tr_t = y.loc[tr_time]
y_va_t = y.loc[va_time]
num_med_t, cat_fill_t = imputation_stats(X_tr_raw_t, cat_cols, num_cols)
X_tr_t, cat_map_t = prepare_for_rf(X_tr_raw_t, cat_cols, num_cols, num_med_t, cat_fill_t, cat_mapping=None)
X_va_t = prepare_for_rf(X_va_raw_t, cat_cols, num_cols, num_med_t, cat_fill_t, cat_mapping=cat_map_t)
y_tr_log_t = np.log1p(y_tr_t.values)
search_time = RandomizedSearchCV(
    estimator=rf_base,
    param_distributions=rf_param_dist,
    n_iter=8,
    scoring="neg_root_mean_squared_error",
    cv=3,
    random_state=RNG,
    n_jobs=-1,
    verbose=1,
)
search_time.fit(X_tr_t, y_tr_log_t)
model_time = search_time.best_estimator_
pred_va_t = np.expm1(model_time.predict(X_va_t))
rmse_t = root_mean_squared_error(y_va_t, pred_va_t)
mae_t = mean_absolute_error(y_va_t, pred_va_t)
print(f"Time split: train {len(X_tr_t):,}, val {len(X_va_t):,} (val >= {cutoff.date()})")
print(f"Time-split validation RMSE (dollars): {rmse_t:,.2f}")
print(f"Time-split validation MAE (dollars): {mae_t:,.2f}")
print("time best_params_:", search_time.best_params_)


Fitting 3 folds for each of 8 candidates, totalling 24 fits


KeyboardInterrupt: 

In [7]:
# --- B) Random 80% / 20% validation + tuning ---
X_tr_raw_r, X_va_raw_r, y_tr_r, y_va_r = train_test_split(
    X_train,
    y,
    test_size=0.20,
    random_state=RNG,
    shuffle=True,
)
num_med_r, cat_fill_r = imputation_stats(X_tr_raw_r, cat_cols, num_cols)
X_tr_r, cat_map_r = prepare_for_rf(X_tr_raw_r, cat_cols, num_cols, num_med_r, cat_fill_r, cat_mapping=None)
X_va_r = prepare_for_rf(X_va_raw_r, cat_cols, num_cols, num_med_r, cat_fill_r, cat_mapping=cat_map_r)
y_tr_log_r = np.log1p(y_tr_r.values)
search_rand = RandomizedSearchCV(
    estimator=rf_base,
    param_distributions=rf_param_dist,
    n_iter=8,
    scoring="neg_root_mean_squared_error",
    cv=3,
    random_state=RNG,
    n_jobs=-1,
    verbose=1,
)
search_rand.fit(X_tr_r, y_tr_log_r)
model_rand = search_rand.best_estimator_
pred_va_r = np.expm1(model_rand.predict(X_va_r))
rmse_r = root_mean_squared_error(y_va_r, pred_va_r)
mae_r = mean_absolute_error(y_va_r, pred_va_r)
print(
    f"Random 80/20: train {len(X_tr_r):,} ({100 * len(X_tr_r) / len(X_train):.1f}%), "
    f"val {len(X_va_r):,} ({100 * len(X_va_r) / len(X_train):.1f}%)"
)
print(f"Random-split validation RMSE (dollars): {rmse_r:,.2f}")
print(f"Random-split validation MAE (dollars): {mae_r:,.2f}")
print("random best_params_:", search_rand.best_params_)


Fitting 3 folds for each of 8 candidates, totalling 24 fits
Random 80/20: train 120,507 (80.0%), val 30,127 (20.0%)
Random-split validation RMSE (dollars): 22,806.22
Random-split validation MAE (dollars): 16,389.05
random best_params_: {'n_estimators': 700, 'min_samples_split': 10, 'min_samples_leaf': 1, 'max_features': 0.5, 'max_depth': 24, 'bootstrap': True}


In [ ]:
# --- C) Full train refit for submissions (_t and _r) ---
num_med_f, cat_fill_f = imputation_stats(X_train, cat_cols, num_cols)
X_full, cat_map_f = prepare_for_rf(X_train, cat_cols, num_cols, num_med_f, cat_fill_f, cat_mapping=None)
X_test_p = prepare_for_rf(X_test, cat_cols, num_cols, num_med_f, cat_fill_f, cat_mapping=cat_map_f)
y_log_full = np.log1p(y.values)

model_final_t = RandomForestRegressor(
    **search_time.best_params_,
    random_state=RNG,
    n_jobs=-1,
)
model_final_r = RandomForestRegressor(
    **search_rand.best_params_,
    random_state=RNG,
    n_jobs=-1,
)

model_final_t.fit(X_full, y_log_full)
model_final_r.fit(X_full, y_log_full)

test_pred_t = np.expm1(model_final_t.predict(X_test_p))
test_pred_r = np.expm1(model_final_r.predict(X_test_p))

sample = pd.read_csv(SAMPLE_SUB_PATH, nrows=5)
sub_t = pd.DataFrame({"Id": test["id"], "Predicted": test_pred_t})
sub_r = pd.DataFrame({"Id": test["id"], "Predicted": test_pred_r})
assert list(sub_t.columns) == list(sample.columns)
assert list(sub_r.columns) == list(sample.columns)

SUBMISSION_PATH_T.parent.mkdir(parents=True, exist_ok=True)
sub_t.to_csv(SUBMISSION_PATH_T, index=False)
sub_r.to_csv(SUBMISSION_PATH_R, index=False)
print(f"Wrote {SUBMISSION_PATH_T.resolve()} ({len(sub_t):,} rows)")
print(f"Wrote {SUBMISSION_PATH_R.resolve()} ({len(sub_r):,} rows)")
print(sub_t.head())



## Compare splits

- **Random 80/20** RMSE is often **lower** than the **time-based** RMSE because future rows can leak into training.
- For realistic forecasting behavior, the **time-based** metric is usually the better proxy.

